In [1]:
import pandas as pd
import yfinance as yf
from datetime import datetime
# We use curl_cffi instead of requests to perfectly mimic a real Chrome browser
from curl_cffi import requests 

# Set up browser headers that match real Chrome behavior on a main page visit
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/'
}

def get_announcements(symbol, years=5):
    # Use 'chrome' impersonation to match the TLS/JA3 fingerprint of a real browser
    session = requests.Session(impersonate="chrome")
    
    try:
        # Step 1: Hit the home page first to collect cookies naturally
        session.get("https://www.nseindia.com", headers=HEADERS, timeout=10)
        
        # Step 2: Hit the actual corporate announcement API endpoint
        url = f"https://www.nseindia.com/api/corporate-announcements?symbol={symbol}"
        r = session.get(url, headers=HEADERS, timeout=10)
        
        if r.status_code != 200:
            print(f"  ↳ NSE blocked request for {symbol} (Status Code: {r.status_code})")
            return pd.DataFrame(columns=["symbol", "subject", "date"])
            
        data = r.json()
    except Exception as e:
        print(f"  ↳ Network or JSON parsing error for {symbol}: {e}")
        return pd.DataFrame(columns=["symbol", "subject", "date"])

    # Filter out anything that doesn't mention Quarterly Results
    results = [item for item in data if "Quarterly Results" in item.get("subject", "")]
    df = pd.DataFrame(results)

    if not df.empty:
        # Standardize dates for matching
        df["date"] = pd.to_datetime(df["dt"], errors="coerce").dt.date
        cutoff = (pd.Timestamp.today() - pd.DateOffset(years=years)).date()
        df = df[df["date"] >= cutoff]
        return df[["symbol", "subject", "date"]]
    else:
        return pd.DataFrame(columns=["symbol", "subject", "date"])


def merge_price_with_earnings(symbol, yf_symbol, start="2020-01-01", end="2026-01-01"):
    # Download reliable historical price data from Yahoo (perfectly fine for prices)
    prices = yf.download(yf_symbol, start=start, end=end, progress=False)
    
    if prices.empty:
        raise ValueError(f"No price data returned for {yf_symbol}")
        
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)

    # Pull the exact corporate event dates directly from official NSE
    print(f"Fetching official NSE announcements for {symbol}...")
    announcements_df = get_announcements(symbol)
    earning_dates = announcements_df["date"].tolist()

    # Clean the index timeframes to match pure dates
    prices.index = pd.to_datetime(prices.index).date
    
    # Flag the days matching the official NSE list
    prices['EarningsFlag'] = prices.index.isin(earning_dates)
    return prices


# Running your specific dictionary setup
nifty50 = {
    "RELIANCE": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFCBANK": "HDFCBANK.NS",
    "INFY": "INFY.NS",
    "ICICIBANK": "ICICIBANK.NS"
}

combined_data = {}
for nse_symbol, yf_symbol in nifty50.items():
    try:
        df = merge_price_with_earnings(nse_symbol, yf_symbol)
        combined_data[nse_symbol] = df
        print(f"Successfully processed {nse_symbol}\n")
    except Exception as e:
        print(f"Error processing {nse_symbol}: {e}\n")

# Access the dictionary safely
if "RELIANCE" in combined_data:
    print("--- Official NSE Earnings Match (Reliance Preview) ---")
    print(combined_data["RELIANCE"].head())
    
    # Filter to show your flag working on official dates
    nse_flagged_days = combined_data["RELIANCE"][combined_data["RELIANCE"]['EarningsFlag'] == True]
    print(f"\nFound {len(nse_flagged_days)} corporate announcement dates on the price charts.")
    print(nse_flagged_days.head())

Fetching official NSE announcements for RELIANCE...
  ↳ Network or JSON parsing error for RELIANCE: Expecting value: line 1 column 1 (char 0)
Successfully processed RELIANCE

Fetching official NSE announcements for TCS...
  ↳ Network or JSON parsing error for TCS: Expecting value: line 1 column 1 (char 0)
Successfully processed TCS

Fetching official NSE announcements for HDFCBANK...
  ↳ Network or JSON parsing error for HDFCBANK: Expecting value: line 1 column 1 (char 0)
Successfully processed HDFCBANK

Fetching official NSE announcements for INFY...
  ↳ Network or JSON parsing error for INFY: Expecting value: line 1 column 1 (char 0)
Successfully processed INFY

Fetching official NSE announcements for ICICIBANK...
  ↳ Network or JSON parsing error for ICICIBANK: Expecting value: line 1 column 1 (char 0)
Successfully processed ICICIBANK

--- Official NSE Earnings Match (Reliance Preview) ---
Price            Close        High         Low        Open    Volume  \
2020-01-01  675.324158

In [2]:
import pandas as pd
import yfinance as yf
from datetime import datetime
from curl_cffi import requests 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/'
}

def get_announcements(symbol, years=5):
    session = requests.Session(impersonate="chrome")
    try:
        session.get("https://www.nseindia.com", headers=HEADERS, timeout=10)
        url = f"https://www.nseindia.com/api/corporate-announcements?symbol={symbol}"
        r = session.get(url, headers=HEADERS, timeout=10)
        
        if r.status_code != 200:
            return pd.DataFrame(columns=["symbol", "subject", "date"])
            
        data = r.json()
    except Exception as e:
        print(f"  ↳ Network/API Error for {symbol}: {e}")
        return pd.DataFrame(columns=["symbol", "subject", "date"])

    # Filter to only capture Quarterly Results filings
    results = [item for item in data if "Quarterly Results" in item.get("subject", "")]
    df = pd.DataFrame(results)

    if not df.empty:
        # Crucial fix: Parse the entire timestamp string cleanly
        df["date"] = pd.to_datetime(df["dt"], errors="coerce")
        cutoff = pd.Timestamp.today() - pd.DateOffset(years=years)
        df = df[df["date"] >= cutoff]
        return df[["symbol", "subject", "date"]]
    else:
        return pd.DataFrame(columns=["symbol", "subject", "date"])


def merge_price_with_earnings(symbol, yf_symbol, start="2020-01-01", end="2026-01-01"):
    prices = yf.download(yf_symbol, start=start, end=end, progress=False)
    
    if prices.empty:
        raise ValueError(f"No price data returned for {yf_symbol}")
        
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)

    # Initialize the earnings flag to False for all days
    prices['EarningsFlag'] = False

    # Pull the exact corporate event dates from the official NSE API
    print(f"Fetching official NSE announcements for {symbol}...")
    announcements_df = get_announcements(symbol)
    
    if announcements_df.empty:
        print(f"  ↳ No corporate announcements found on NSE for {symbol}.")
        return prices

    # Clean price index to timezone-naive timestamps for accurate calculations
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    announcement_dates = pd.to_datetime(announcements_df["date"]).dt.tz_localize(None)

    # Smart Matching Loop: Finds the exact trading session that absorbed the news
    matched_count = 0
    for announce_date in announcement_dates:
        # Find trading days that are greater than or equal to the announcement timestamp
        future_trading_days = prices.index[prices.index >= announce_date]
        
        if not future_trading_days.empty:
            # The very first available trading day in that list is our market match
            actual_trading_day = future_trading_days[0]
            prices.at[actual_trading_day, 'EarningsFlag'] = True
            matched_count += 1

    print(f"  ↳ Aligned {matched_count} announcements to active market sessions.")
    return prices


# Execution Block
nifty50 = {
    "RELIANCE": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFCBANK": "HDFCBANK.NS"
}

combined_data = {}
for nse_symbol, yf_symbol in nifty50.items():
    try:
        df = merge_price_with_earnings(nse_symbol, yf_symbol)
        combined_data[nse_symbol] = df
        print(f"Successfully processed {nse_symbol}\n")
    except Exception as e:
        print(f"Error processing {nse_symbol}: {e}\n")

# Access and show structural preview 
if "RELIANCE" in combined_data:
    print("--- Aligned NSE Earnings Match (Reliance Preview) ---")
    earnings_days = combined_data["RELIANCE"][combined_data["RELIANCE"]['EarningsFlag'] == True]
    print(earnings_days[["Open", "High", "Low", "Close", "Volume", "EarningsFlag"]].head())

Fetching official NSE announcements for RELIANCE...
  ↳ Network/API Error for RELIANCE: Expecting value: line 1 column 1 (char 0)
  ↳ No corporate announcements found on NSE for RELIANCE.
Successfully processed RELIANCE

Fetching official NSE announcements for TCS...
  ↳ Network/API Error for TCS: Expecting value: line 1 column 1 (char 0)
  ↳ No corporate announcements found on NSE for TCS.
Successfully processed TCS

Fetching official NSE announcements for HDFCBANK...
  ↳ Network/API Error for HDFCBANK: Expecting value: line 1 column 1 (char 0)
  ↳ No corporate announcements found on NSE for HDFCBANK.
Successfully processed HDFCBANK

--- Aligned NSE Earnings Match (Reliance Preview) ---
Empty DataFrame
Columns: [Open, High, Low, Close, Volume, EarningsFlag]
Index: []


In [3]:
import pandas as pd
import yfinance as yf
from datetime import datetime
from curl_cffi import requests 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/companies-listing/corporate-filings-announcements'
}

def get_announcements(symbol, years=5):
    session = requests.Session(impersonate="chrome")
    try:
        # Crucial: Visit the filings portal page to get specialized session cookies
        session.get("https://www.nseindia.com/companies-listing/corporate-filings-announcements", headers=HEADERS, timeout=10)
        
        # Pull data from the endpoint
        url = f"https://www.nseindia.com/api/corporate-announcements?symbol={symbol}"
        r = session.get(url, headers=HEADERS, timeout=10)
        
        if r.status_code != 200:
            print(f"  ↳ NSE Endpoint returned status code {r.status_code}")
            return pd.DataFrame()
            
        data = r.json()
        
        # If the response is wrapped inside a dictionary like {"data": [...]}
        if isinstance(data, dict) and "data" in data:
            data = data["data"]
            
    except Exception as e:
        print(f"  ↳ Connection failed for {symbol}: {e}")
        return pd.DataFrame()

    if not data or not isinstance(data, list):
        print(f"  ↳ No list data found in response for {symbol}")
        return pd.DataFrame()

    # Filter to look for Financial results or Board Meetings discussing results
    valid_records = []
    for item in data:
        subject = str(item.get("subject", "")).upper()
        desc = str(item.get("desc", "")).upper()
        
        # Broaden filter text slightly to catch alternative phrasing like 'Financial Results'
        if "QUARTERLY" in subject or "FINANCIAL RESULTS" in subject or "QUARTERLY" in desc:
            
            # FAIL-SAFE: Check every known naming key used by NSE for timestamps
            potential_date = None
            for key in ["attchblddate", "an_dt", "dt", "p_dt", "date"]:
                if key in item and item[key]:
                    potential_date = item[key]
                    break
            
            if potential_date:
                valid_records.append({
                    "symbol": item.get("symbol", symbol),
                    "subject": item.get("subject", "Quarterly Results"),
                    "raw_date_string": potential_date
                })

    df = pd.DataFrame(valid_records)

    if not df.empty:
        # Parse the raw date text flexibly
        df["date"] = pd.to_datetime(df["raw_date_string"], errors="coerce")
        # Drop rows where the date couldn't parse
        df = df.dropna(subset=["date"])
        
        # Filter out anything older than your cutoff range
        cutoff = pd.Timestamp.today() - pd.DateOffset(years=years)
        df = df[df["date"] >= cutoff]
        return df[["symbol", "subject", "date"]]
        
    return pd.DataFrame()


def merge_price_with_earnings(symbol, yf_symbol, start="2020-01-01", end="2026-01-01"):
    prices = yf.download(yf_symbol, start=start, end=end, progress=False)
    
    if prices.empty:
        raise ValueError(f"No price data loaded from Yahoo for {yf_symbol}")
        
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)

    prices['EarningsFlag'] = False

    print(f"Fetching structural NSE data for {symbol}...")
    announcements_df = get_announcements(symbol)
    
    if announcements_df.empty:
        print(f"  ↳ Warning: 0 filtered records returned by the NSE parser for {symbol}.")
        return prices

    # Match timezone boundaries
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    announcement_dates = pd.to_datetime(announcements_df["date"]).dt.tz_localize(None)

    matched_count = 0
    for announce_date in announcement_dates:
        # Map the exact announcement time directly forward to the next open trading block
        future_trading_days = prices.index[prices.index >= announce_date]
        if not future_trading_days.empty:
            actual_trading_day = future_trading_days[0]
            prices.at[actual_trading_day, 'EarningsFlag'] = True
            matched_count += 1

    print(f"  ↳ Successfully pinned {matched_count} announcements onto chart timelines.")
    return prices


# Execution Context
nifty50 = {
    "RELIANCE": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFCBANK": "HDFCBANK.NS"
}

combined_data = {}
for nse_symbol, yf_symbol in nifty50.items():
    try:
        df = merge_price_with_earnings(nse_symbol, yf_symbol)
        combined_data[nse_symbol] = df
        print(f"Finished processing {nse_symbol}\n")
    except Exception as e:
        print(f"Error handling loops for {nse_symbol}: {e}\n")

# Verify structural population
if "RELIANCE" in combined_data:
    r_df = combined_data["RELIANCE"]
    earnings_days = r_df[r_df['EarningsFlag'] == True]
    
    if not earnings_days.empty:
        print("--- Verified Output Rows ---")
        print(earnings_days[["Open", "High", "Low", "Close", "EarningsFlag"]].head())
    else:
        print("--- Fallback Notice ---")
        print("No matches hit. Here is the head of your standard chart data:")
        print(r_df[["Open", "Close", "EarningsFlag"]].head())

Fetching structural NSE data for RELIANCE...
  ↳ Connection failed for RELIANCE: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: 0 filtered records returned by the NSE parser for RELIANCE.
Finished processing RELIANCE

Fetching structural NSE data for TCS...
  ↳ Connection failed for TCS: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: 0 filtered records returned by the NSE parser for TCS.
Finished processing TCS

Fetching structural NSE data for HDFCBANK...
  ↳ Connection failed for HDFCBANK: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: 0 filtered records returned by the NSE parser for HDFCBANK.
Finished processing HDFCBANK

--- Fallback Notice ---
No matches hit. Here is the head of your standard chart data:
Price             Open       Close  EarningsFlag
Date                                            
2020-01-01  679.081936  675.324158         False
2020-01-02  676.397899  686.821228         False
2020-01-03  685.792252  687.648804         False
2020-01-06 

In [4]:
import pandas as pd
import yfinance as yf
from datetime import datetime
from curl_cffi import requests 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/companies-listing/corporate-filings-announcements'
}

def get_announcements(symbol, years=5):
    session = requests.Session(impersonate="chrome")
    try:
        # Step 1: Collect structural cookies from the main filings directory page
        session.get("https://www.nseindia.com/companies-listing/corporate-filings-announcements", headers=HEADERS, timeout=10)
        
        # Step 2: Grab the JSON data structure
        url = f"https://www.nseindia.com/api/corporate-announcements?symbol={symbol}"
        r = session.get(url, headers=HEADERS, timeout=10)
        
        if r.status_code != 200:
            print(f"  ↳ NSE Blocked Endpoint (HTTP Status {r.status_code})")
            return pd.DataFrame()
            
        raw_json = r.json()
        
        # CRUCIAL FIX: NSE wraps data arrays inside the dictionary key labeled 'rows'
        data_list = []
        if isinstance(raw_json, list):
            data_list = raw_json
        elif isinstance(raw_json, dict):
            if "rows" in raw_json:
                data_list = raw_json["rows"]
            elif "data" in raw_json:
                data_list = raw_json["data"]
                
        print(f"  ↳ Total raw announcements received from NSE for {symbol}: {len(data_list)}")
        
    except Exception as e:
        print(f"  ↳ Connection failed for {symbol}: {e}")
        return pd.DataFrame()

    if not data_list:
        return pd.DataFrame()

    valid_records = []
    for item in data_list:
        # Normalize the strings to upper case to remove case variations
        subject = str(item.get("subject", "")).upper()
        desc = str(item.get("desc", "")).upper()
        
        # Catch all common combinations of financial result descriptions
        if any(keyword in subject or keyword in desc for keyword in ["QUARTERLY", "FINANCIAL RESULT", "FINANCIALS", "EARNING"]):
            
            # Extract timestamps matching corporate dissemination fields
            potential_date = None
            for key in ["attchblddate", "an_dt", "dt", "p_dt", "date"]:
                if key in item and item[key]:
                    potential_date = item[key]
                    break
            
            if potential_date:
                valid_records.append({
                    "symbol": item.get("symbol", symbol),
                    "subject": item.get("subject", "Financial Results"),
                    "raw_date_string": potential_date
                })

    df = pd.DataFrame(valid_records)
    print(f"  ↳ Filtered down to {len(df)} financial/earnings records.")

    if not df.empty:
        # Standardize date objects cleanly
        df["date"] = pd.to_datetime(df["raw_date_string"], errors="coerce")
        df = df.dropna(subset=["date"])
        
        cutoff = pd.Timestamp.today() - pd.DateOffset(years=years)
        df = df[df["date"] >= cutoff]
        return df[["symbol", "subject", "date"]]
        
    return pd.DataFrame()


def merge_price_with_earnings(symbol, yf_symbol, start="2020-01-01", end="2026-01-01"):
    prices = yf.download(yf_symbol, start=start, end=end, progress=False)
    
    if prices.empty:
        raise ValueError(f"No price data loaded from Yahoo for {yf_symbol}")
        
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)

    # Initialize all rows to False
    prices['EarningsFlag'] = False

    print(f"Processing official NSE data stream for {symbol}...")
    announcements_df = get_announcements(symbol)
    
    if announcements_df.empty:
        print(f"  ↳ Warning: No parsed corporate matches found for {symbol}.")
        return prices

    # Clean local indices to timezone-naive formats
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    announcement_dates = pd.to_datetime(announcements_df["date"]).dt.tz_localize(None)

    matched_count = 0
    for announce_date in announcement_dates:
        # Handle calendar adjustments (roll forward to next available market hour entry)
        future_trading_days = prices.index[prices.index >= announce_date]
        if not future_trading_days.empty:
            actual_trading_day = future_trading_days[0]
            prices.at[actual_trading_day, 'EarningsFlag'] = True
            matched_count += 1

    print(f"  ↳ Successfully flagged {matched_count} dates on the chart vector.")
    return prices


# Execution Area
nifty50 = {
    "RELIANCE": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFCBANK": "HDFCBANK.NS"
}

combined_data = {}
for nse_symbol, yf_symbol in nifty50.items():
    try:
        df = merge_price_with_earnings(nse_symbol, yf_symbol)
        combined_data[nse_symbol] = df
        print(f"Completed processing for {nse_symbol}\n")
    except Exception as e:
        print(f"Error handling loops for {nse_symbol}: {e}\n")

# Output Verification View
if "RELIANCE" in combined_data:
    r_df = combined_data["RELIANCE"]
    earnings_days = r_df[r_df['EarningsFlag'] == True]
    
    if not earnings_days.empty:
        print("--- Verified Output Rows ---")
        print(earnings_days[["Open", "High", "Low", "Close", "EarningsFlag"]].head())
    else:
        print("--- Structural Fallback Notice ---")
        print("Matches still hit zero. Here are the column listings present in your dataframe structure:")
        print(r_df.columns.tolist())
        print(r_df.head(2))

Processing official NSE data stream for RELIANCE...
  ↳ Connection failed for RELIANCE: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: No parsed corporate matches found for RELIANCE.
Completed processing for RELIANCE

Processing official NSE data stream for TCS...
  ↳ Connection failed for TCS: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: No parsed corporate matches found for TCS.
Completed processing for TCS

Processing official NSE data stream for HDFCBANK...
  ↳ Connection failed for HDFCBANK: Expecting value: line 1 column 1 (char 0)
  ↳ Warning: No parsed corporate matches found for HDFCBANK.
Completed processing for HDFCBANK

--- Structural Fallback Notice ---
Matches still hit zero. Here are the column listings present in your dataframe structure:
['Close', 'High', 'Low', 'Open', 'Volume', 'EarningsFlag']
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2020-01-01  67

In [5]:
import io
import pandas as pd
import yfinance as yf
from datetime import datetime
from curl_cffi import requests

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.nseindia.com/companies-listing/corporate-filings-announcements'
}

def fetch_master_nse_announcements(years=5):
    """
    Downloads the entire recent historical corporate announcements database 
    from the NSE master CSV export endpoint in a single query.
    """
    print("Connecting to NSE to download master corporate filings directory...")
    session = requests.Session(impersonate="chrome")
    
    try:
        # Step 1: Establish real-user cookie session on the entry board
        session.get("https://www.nseindia.com/companies-listing/corporate-filings-announcements", headers=HEADERS, timeout=15)
        
        # Step 2: Request the massive compiled database dump directly via CSV export
        csv_url = "https://www.nseindia.com/api/corporate-announcements?index=equities&csv=true"
        response = session.get(csv_url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            print(f"  ↳ Master CSV download rejected by NSE (Status Code: {response.status_code})")
            return pd.DataFrame()
            
        # Parse text string into Pandas dataframe structure
        csv_data = response.content.decode('utf-8-sig', errors='ignore')
        df = pd.read_csv(io.StringIO(csv_data))
        print(f"  ↳ Successfully parsed {len(df)} total corporate records across all NSE listings.")
        
        # Clean down the column naming spaces if present
        df.columns = df.columns.str.strip().str.upper()
        
        # Look for standard dates column mappings
        date_col = None
        for col in ['BROADCAST DATE/TIME', 'BROADCAST DATE', 'DATE', 'AN_DT']:
            if col in df.columns:
                date_col = col
                break
                
        if not date_col:
            print("  ↳ Could not find valid date headers in the master registry dump.")
            return pd.DataFrame()
            
        df['PARSED_DATE'] = pd.to_datetime(df[date_col], errors='coerce')
        df = df.dropna(subset=['PARSED_DATE'])
        
        # Slice off anything outside the historical range window
        cutoff = pd.Timestamp.today() - pd.DateOffset(years=years)
        df = df[df['PARSED_DATE'] >= cutoff]
        
        return df

    except Exception as e:
        print(f"  ↳ Failed to download or process master list: {e}")
        return pd.DataFrame()


def merge_price_with_earnings(symbol, yf_symbol, master_announcements_df, start="2020-01-01", end="2026-01-01"):
    """Loads historical prices and slices the locally stored master table for flags"""
    prices = yf.download(yf_symbol, start=start, end=end, progress=False)
    
    if prices.empty:
        raise ValueError(f"No price data loaded from Yahoo for {yf_symbol}")
        
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)

    prices['EarningsFlag'] = False
    prices.index = pd.to_datetime(prices.index).tz_localize(None)

    if master_announcements_df.empty:
        return prices

    # Filter out company records just for this specific symbol loop locally
    company_mask = (master_announcements_df['SYMBOL'].astype(str).str.strip().str.upper() == symbol.upper())
    company_df = master_announcements_df[company_mask]
    
    # Isolate financial result entries strictly
    results_mask = company_df['SUBJECT'].astype(str).str.upper().str.contains('QUARTERLY|FINANCIAL RESULT|FINANCIALS|EARNING', na=False)
    earnings_df = company_df[results_mask]

    if earnings_df.empty:
        print(f"  ↳ No historical quarterly filings noted for {symbol} within the active slice.")
        return prices

    # Match timeframes up to the active financial sessions
    announcement_dates = earnings_df['PARSED_DATE'].dt.tz_localize(None)
    matched_count = 0
    
    for announce_date in announcement_dates:
        future_trading_days = prices.index[prices.index >= announce_date]
        if not future_trading_days.empty:
            actual_trading_day = future_trading_days[0]
            prices.at[actual_trading_day, 'EarningsFlag'] = True
            matched_count += 1

    print(f"  ↳ Aligned {matched_count} official corporate filings with your stock chart index.")
    return prices


# Main Loop Area
nifty50 = {
    "RELIANCE": "RELIANCE.NS",
    "TCS": "TCS.NS",
    "HDFCBANK": "HDFCBANK.NS"
}

# Fetch the entire stock exchange filing record pool ONCE
master_df = fetch_master_nse_announcements(years=5)

combined_data = {}
for nse_symbol, yf_symbol in nifty50.items():
    try:
        print(f"Processing calculations for {nse_symbol}...")
        df = merge_price_with_earnings(nse_symbol, yf_symbol, master_df)
        combined_data[nse_symbol] = df
        print(f"Completed mapping data metrics for {nse_symbol}.\n")
    except Exception as e:
        print(f"Error handling math transformations for {nse_symbol}: {e}\n")

# Verify structural outputs match 
if "RELIANCE" in combined_data:
    r_df = combined_data["RELIANCE"]
    earnings_days = r_df[r_df['EarningsFlag'] == True]
    
    if not earnings_days.empty:
        print("\n=== Verified Output Rows ===")
        print(earnings_days[["Open", "High", "Low", "Close", "EarningsFlag"]].head())
    else:
        print("\n=== Data Mismatch Notice ===")
        print("Prices generated cleanly but flags remained unassigned. Sample price rows:")
        print(r_df[["Open", "Close", "EarningsFlag"]].head(3))

Connecting to NSE to download master corporate filings directory...
  ↳ Successfully parsed 20 total corporate records across all NSE listings.
Processing calculations for RELIANCE...
  ↳ No historical quarterly filings noted for RELIANCE within the active slice.
Completed mapping data metrics for RELIANCE.

Processing calculations for TCS...
  ↳ No historical quarterly filings noted for TCS within the active slice.
Completed mapping data metrics for TCS.

Processing calculations for HDFCBANK...
  ↳ No historical quarterly filings noted for HDFCBANK within the active slice.
Completed mapping data metrics for HDFCBANK.


=== Data Mismatch Notice ===
Prices generated cleanly but flags remained unassigned. Sample price rows:
Price             Open       Close  EarningsFlag
Date                                            
2020-01-01  679.081936  675.324158         False
2020-01-02  676.397899  686.821228         False
2020-01-03  685.792252  687.648804         False
